# Lesson 2 — Philidor Defense (as White)
**Project 1500 | Priority #2**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

In [ ]:
import chess
import chess.pgn
import chess.svg
import json
import glob
import io
import re
from collections import Counter, defaultdict
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE):
    svg = chess.svg.board(board, arrows=arrows or [], size=size)
    if caption:
        display(HTML(f'<p style="font-family:monospace;font-size:13px;margin:4px 0 8px 0;color:#aaa">{caption}</p>'))
    display(SVG(data=svg))

## Step 1 — Load rapid games as white (April 2025+)

In [ ]:
def load_rapid_games_as_white(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(glob.glob(f'{games_dir}/2025_*.json') + glob.glob(f'{games_dir}/2026_*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            white = g.get('white', {})
            if white.get('username', '').lower() != username.lower(): continue
            result = white.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_white_games = load_rapid_games_as_white(USERNAME)
print(f'Loaded {len(all_white_games)} rapid games as white')

## Step 2 — Filter to Philidor games and compute stats

We detect Philidor games by matching the ECO URL (Chess.com's opening classification).
We also track the critical junction: after you push d4 and black takes, do you recapture with **Nxd4** or **Qxd4**?

In [ ]:
PHILIDOR_PATTERN = 'Philidor'

philidor_games = [
    g for g in all_white_games
    if PHILIDOR_PATTERN in g['eco_url']
]

counts = Counter(g['outcome'] for g in philidor_games)
total  = len(philidor_games)
wr     = 100 * counts['win'] / total if total else 0

print(f'Philidor games as white: {total}')
print(f'W:{counts["win"]}  L:{counts["loss"]}  D:{counts["draw"]}  Win%: {wr:.1f}%')

if total < 15:
    print('\nNote: sample size is small — treat percentages as directional, not definitive.')

# Junction: how do you recapture after ...exd4?
# Detect by looking for Nxd4 or Qxd4 after a d4 push in the game
recap_stats = defaultdict(Counter)

for g in philidor_games:
    try:
        game  = chess.pgn.read_game(io.StringIO(g['pgn']))
        board = game.board()
        moves = list(game.mainline_moves())
        d4_played = False
        for i, move in enumerate(moves[:20]):
            if i % 2 == 0:  # white's move
                san = board.san(move)
                if san == 'd4': d4_played = True
                if d4_played and san in ('Nxd4', 'Qxd4'):
                    recap_stats[san][g['outcome']] += 1
                    break
            board.push(move)
    except Exception:
        pass

print('\nRecapture stats after ...exd4:')
print(f'{"Move":<8} {"W":>4} {"L":>4} {"D":>3} {"Total":>6} {"Win%":>7}')
print('-' * 38)
for move in ['Nxd4', 'Qxd4']:
    c = recap_stats[move]
    t = sum(c.values())
    if t == 0: continue
    print(f'{move:<8} {c["win"]:>4} {c["loss"]:>4} {c["draw"]:>3} {t:>6} {100*c["win"]/t:>7.1f}%')

## Step 3 — Recapture comparison chart

In [ ]:
moves_to_plot = ['Nxd4', 'Qxd4']
wrs    = [100 * recap_stats[m]['win'] / max(sum(recap_stats[m].values()), 1) for m in moves_to_plot]
totals = [sum(recap_stats[m].values()) for m in moves_to_plot]
colors = ['#27ae60' if wr >= 45 else '#c0392b' for wr in wrs]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(moves_to_plot, wrs, color=colors, width=0.4, edgecolor='white')
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.set_ylabel('Win %')
ax.set_title('Philidor: recapture on d4 — Nxd4 vs Qxd4 (computed from your games)')
ax.set_ylim(0, 85)
for bar, wr, n in zip(bars, wrs, totals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{wr:.0f}%\n({n}g)', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## Step 4 — Board diagrams *(illustrative example lines)*

These diagrams use example positions — the statistics came from the analysis above.

In [ ]:
# [ILLUSTRATIVE] Position after exd4 — showing the recapture choice
# Using a line without ...c6 so Nc6 is available as black's response to Qxd4
BEFORE_RECAP = ['e4','e5','Nf3','d6','Bc4','Nf6','Nc3','Be7','d4','exd4']

show_board(
    board_after(BEFORE_RECAP),
    arrows=[
        chess.svg.Arrow(chess.F3, chess.D4, color='#27ae60'),  # Nxd4 — good
        chess.svg.Arrow(chess.D1, chess.D4, color='#c0392b'),  # Qxd4 — bad
    ],
    caption='[ILLUSTRATIVE] After exd4 — Green=Nxd4 (knight to centre) | Red=Qxd4 (queen harassed)'
)

In [ ]:
# [ILLUSTRATIVE] Why Qxd4 fails — queen gets harassed by Nc6 then Be6
show_board(
    board_after(BEFORE_RECAP + ['Qxd4','Nc6','Qd1','Be6']),
    arrows=[
        chess.svg.Arrow(chess.C6, chess.D4, color='#c0392b'),  # Nc6 attacked the queen
        chess.svg.Arrow(chess.E6, chess.B3, color='#c0392b'),  # Be6 pressures
    ],
    caption='[ILLUSTRATIVE] After Qxd4 Nc6 Qd1 Be6 — black developed two pieces for free while queen shuffled back'
)

In [ ]:
# [ILLUSTRATIVE] The correct recapture — Nxd4 dominates the centre
show_board(
    board_after(BEFORE_RECAP + ['Nxd4']),
    arrows=[
        chess.svg.Arrow(chess.D4, chess.C6, color='#27ae60'),  # Nd4 threatens
        chess.svg.Arrow(chess.C4, chess.F7, color='#2980b9'),  # Bc4 eyes f7
    ],
    caption='[ILLUSTRATIVE] After Nxd4 — knight controls the centre, Bc4 eyes f7, white is clearly better'
)

## Step 5 — Summary

In [ ]:
nxd4_wr = 100 * recap_stats['Nxd4']['win'] / max(sum(recap_stats['Nxd4'].values()), 1)
qxd4_wr = 100 * recap_stats['Qxd4']['win'] / max(sum(recap_stats['Qxd4'].values()), 1)

print('PHILIDOR DEFENSE — SUMMARY')
print('=' * 50)
print(f'\nTotal Philidor games as white: {total}  ({wr:.1f}% win rate)')
print(f'\nAfter ...exd4:')
print(f'  Nxd4: {nxd4_wr:.1f}% win ({sum(recap_stats["Nxd4"].values())} games)')
print(f'  Qxd4: {qxd4_wr:.1f}% win ({sum(recap_stats["Qxd4"].values())} games)')
print()
print('RULES:')
print('  1. Play d4 as early as the position allows')
print('  2. After ...exd4, ALWAYS recapture with Nxd4 — never Qxd4')
print('  3. Qxd4 brings the queen out early — it gets chased by ...Nc6 and ...Be6')
print('  4. d3 instead of d4 is passive — Bc4 has no purpose without the d4 break')